# NLP Evaluation Metrics — runnable companion

The capstone of the NLP arc: **how do you score language output when there is no single right answer?** This notebook reproduces every number on the page and proves the central claims by assertion — each cell `assert`s its point *before* it prints, so a wrong number fails loudly rather than misleading you.

Every metric here is **imported from the single source of truth** `nlp_evaluation.py` (the same module the page's figures are built from), so the prose, the figures, and this notebook cannot drift apart. The core is pure **numpy/stdlib** — it runs on CPU anywhere, instantly, and is fully seeded. We cross-check the from-scratch BLEU/ROUGE against `sacrebleu` and `rouge-score` at the end.

**What we build, in order:** the metric *zoo* (five metrics disagree on one pair) → the *meaning-blindness* punchline (a paraphrase scores low on overlap, high on meaning) → the *bootstrap* significance test (a 'win' that isn't) → *metric-vs-human correlation* (the meta-metric) → the *library cross-check*.

## 0. Environment — honest device + version line
Pure numpy on CPU. We print the versions so the run is reproducible.

In [1]:
import platform
import numpy as np

print('python:', platform.python_version())
print('numpy :', np.__version__)
print('device: CPU (pure numpy / stdlib — no accelerator needed)')
try:
    import torch
    print('torch :', torch.__version__, '(present, not required by the core)')
except ImportError:
    print('torch : not imported (not needed)')

python: 3.12.13
numpy : 2.4.6
device: CPU (pure numpy / stdlib — no accelerator needed)


torch : 2.12.0 (present, not required by the core)


## 1. The QA pair: Exact Match vs token-F1 (Example A)

Extractive QA (SQuAD-style) reports **two** numbers because they disagree. *Exact Match* is 1 only if the strings match exactly; *token-F1* treats each answer as a **bag of tokens** and gives partial credit. The prediction `'Denver Broncos'` against gold `'The Denver Broncos'` is mostly right — so EM is a harsh **0** while token-F1 is a forgiving **0.80**. That gap is the whole reason SQuAD reports both.

In [2]:
from nlp_evaluation import exact_match, token_f1, QA_PRED, QA_GOLD

em = exact_match(QA_PRED, QA_GOLD)
f1 = token_f1(QA_PRED, QA_GOLD)

# Assert the point BEFORE printing: strict EM fails, forgiving F1 gives partial credit.
assert em == 0, 'EM should be 0 — the strings differ by "The"'
assert abs(f1 - 0.80) < 1e-9, f'token-F1 should be exactly 0.80, got {f1}'

print(f'pred = {QA_PRED!r}')
print(f'gold = {QA_GOLD!r}')
print(f'Exact Match = {em}   token-F1 = {f1:.3f}')
print('-> strict EM=0 but mostly-right F1=0.80 — why SQuAD reports both')

pred = 'Denver Broncos'
gold = 'The Denver Broncos'
Exact Match = 0   token-F1 = 0.800
-> strict EM=0 but mostly-right F1=0.80 — why SQuAD reports both


## 2. BLEU from scratch (Example B)

BLEU is four ideas: **clipped n-gram precision** (orders 1–4), a **geometric mean** of them, and a **brevity penalty** that stands in for the recall BLEU lacks. We import the from-scratch implementation and reproduce the page's worked example to the digit: reference *'the cat sat on the warm mat'*, candidate *'the cat sat on the mat'* → **BLEU 67.3**. The single missing word *'warm'* costs ~12 points through the brevity penalty (0.847).

In [3]:
from nlp_evaluation import sentence_bleu, BLEU_REFERENCE, BLEU_CANDIDATE

b = sentence_bleu(BLEU_CANDIDATE, BLEU_REFERENCE)

# Assert the exact worked-example result before showing the breakdown.
assert abs(b['bleu'] - 67.318) < 0.01, f"BLEU should be 67.318, got {b['bleu']}"
assert b['precisions'][0] == 1.0, 'unigram precision should be 1.0 (all words present)'

print(f'reference: {BLEU_REFERENCE!r}')
print(f'candidate: {BLEU_CANDIDATE!r}')
print(f"p1..p4   = {[round(p, 3) for p in b['precisions']]}")
print(f"geo mean = {b['geo_mean']:.4f}   brevity penalty = {b['bp']:.4f}")
print(f"BLEU     = {b['bleu']:.3f}   (without BP it would be {100*b['geo_mean']:.1f})")

reference: 'the cat sat on the warm mat'
candidate: 'the cat sat on the mat'
p1..p4   = [1.0, 0.8, 0.75, 0.667]
geo mean = 0.7953   brevity penalty = 0.8465
BLEU     = 67.318   (without BP it would be 79.5)


## 3. The metric zoo — five surface metrics, one pair, total disagreement

Here is the capstone idea. We run **exact-match, token-F1, BLEU, ROUGE-L, and chrF** on the *same* candidate/reference pairs, plus a deterministic **semantic** (embedding-overlap) metric. The reference is *'the movie was absolutely fantastic'*. Read the rows: metrics that all claim to 'measure overlap' assign **wildly different scores** to the same text — because each measures overlap at a different level (whole string, bag of tokens, word n-grams, character n-grams, or meaning).

In [4]:
from nlp_evaluation import metric_zoo, semantic_overlap, ZOO_REFERENCE, ZOO_CANDIDATES

print(f'reference: {ZOO_REFERENCE!r}\n')
header = f"{'candidate':12} {'EM':>4} {'Tok-F1':>7} {'BLEU':>6} {'ROUGE-L':>8} {'chrF':>6} {'semantic':>9}"
print(header)
print('-' * len(header))
for name, cand in ZOO_CANDIDATES.items():
    z = metric_zoo(cand, ZOO_REFERENCE)
    sem = semantic_overlap(cand, ZOO_REFERENCE)
    print(f"{name:12} {z['Exact match']:4.0f} {z['Token-F1']:7.1f} {z['BLEU']:6.1f} "
          f"{z['ROUGE-L']:8.1f} {z['chrF']:6.1f} {sem:9.1f}")

reference: 'the movie was absolutely fantastic'

candidate      EM  Tok-F1   BLEU  ROUGE-L   chrF  semantic
----------------------------------------------------------
exact copy    100   100.0  100.0    100.0  100.0     100.0
paraphrase      0    40.0    0.0     40.0   15.3      86.1
negation        0    80.0   66.9     80.0   68.8      75.2
unrelated       0     0.0    0.0      0.0   10.5       9.6


![Metric zoo](../../images/nlpeval_metric_zoo.png)

*The same numbers as bars (built by `make_figures_18.py` from these exact functions): on the **paraphrase** the surface metrics see 'wrong' (BLEU 0) but the semantic metric sees 'same meaning' (86); on the **negation** the surface metrics score high on shared words — and the semantic metric is fooled too.*

## 4. Meaning-blindness — the cross-chapter punchline

Isolate the single most important row. A **faithful paraphrase** *'the film was incredibly wonderful'* shares almost no content words with *'the movie was absolutely fantastic'*, so **BLEU collapses to ~0** — the surface metric confuses *different words* with *wrong*. The embedding metric scores it **high** because the meaning is preserved. This is exactly the case embedding metrics (BERTScore, COMET) were built for.

In [5]:
para = ZOO_CANDIDATES['paraphrase']
para_bleu = metric_zoo(para, ZOO_REFERENCE)['BLEU']
para_sem = semantic_overlap(para, ZOO_REFERENCE)

# Assert the punchline before printing it.
assert para_bleu < 20, f'paraphrase BLEU should be LOW (<20), got {para_bleu}'
assert para_sem > 70, f'paraphrase semantic overlap should be HIGH (>70), got {para_sem}'

print(f'paraphrase: {para!r}')
print(f'reference : {ZOO_REFERENCE!r}')
print(f'BLEU (surface)   = {para_bleu:5.1f}  -> sees "wrong"')
print(f'semantic (meaning) = {para_sem:5.1f}  -> sees "same meaning"')

paraphrase: 'the film was incredibly wonderful'
reference : 'the movie was absolutely fantastic'
BLEU (surface)   =   0.0  -> sees "wrong"
semantic (meaning) =  86.1  -> sees "same meaning"


### ...and the limit: embedding metrics are fooled by negation

The same embedding metric that *rescues* the paraphrase **fails** on negation. *'the movie was absolutely terrible'* has the **opposite** meaning of the reference, yet *'terrible'* and *'fantastic'* appear in similar contexts, so their vectors are close and the metric still scores high. **No automatic metric reliably catches meaning reversal** — a crucial limitation to state out loud.

In [6]:
neg = ZOO_CANDIDATES['negation']
neg_sem = semantic_overlap(neg, ZOO_REFERENCE)

# The embedding family's blind spot: opposite meaning still scores high.
assert neg_sem > 50, 'negation should still score high on the embedding metric (the blind spot)'

print(f'negation  : {neg!r}  (OPPOSITE meaning)')
print(f'reference : {ZOO_REFERENCE!r}')
print(f'semantic  = {neg_sem:5.1f}  -> still HIGH — the metric is fooled')
print('lesson: pair an embedding metric with an NLI/entailment or LLM-judge check for negation/facts')

negation  : 'the movie was absolutely terrible'  (OPPOSITE meaning)
reference : 'the movie was absolutely fantastic'
semantic  =  75.2  -> still HIGH — the metric is fooled
lesson: pair an embedding metric with an NLI/entailment or LLM-judge check for negation/facts


## 5. Is the 'win' real? Paired bootstrap significance (Koehn 2004)

A single number lies. System A scores **+2.86 higher** than system B on average — but is that a real improvement or noise? The **paired bootstrap** answers it: resample the test sentences (with replacement) thousands of times, recompute the mean difference each time, and read off a **95% confidence interval**. If the interval **straddles zero**, the 'win' is not significant. Here it does — reporting only the +2.86 would ship noise as progress.

In [7]:
from nlp_evaluation import make_two_systems, paired_bootstrap_diff

sys_a, sys_b = make_two_systems()
boot = paired_bootstrap_diff(sys_a, sys_b)

# Assert the teaching point: a positive observed lead that is NOT significant.
assert boot['observed_diff'] > 0, 'system A should have a small positive observed lead'
assert not boot['significant'], 'the lead should NOT be significant (CI straddles 0)'
assert boot['ci_low'] < 0 < boot['ci_high'], 'the 95% CI must contain 0'

print(f"observed mean diff (A - B) = {boot['observed_diff']:+.2f}")
print(f"95% CI = [{boot['ci_low']:+.2f}, {boot['ci_high']:+.2f}]")
print(f"significant? {boot['significant']}  (the interval contains 0 -> no real difference)")

observed mean diff (A - B) = +2.86
95% CI = [-1.42, +7.05]
significant? False  (the interval contains 0 -> no real difference)


![Bootstrap CI](../../images/nlpeval_bootstrap_ci.png)

*The bootstrap distribution of (A − B): the observed lead (green) sits inside a 95% interval (shaded) that comfortably includes zero (red dashed). The 'win' is indistinguishable from noise.*

## 6. The meta-metric: correlation with human judgement

Every automatic metric is a **proxy** for a human, so the only thing that makes it trustworthy is **how well it correlates with human scores**. We measure that three ways on a small synthetic set: **Spearman ρ** (correlation of *ranks* — monotone agreement, the right one for 'does the metric order outputs like humans?'), **Pearson r** (linear), and **Kendall τ** (pairwise concordance). The metric tracks humans well (ρ ≈ 0.94) but **not perfectly** — the realistic regime where a metric is good for *ranking systems* yet unreliable on any *single* output.

In [8]:
from nlp_evaluation import make_correlation_data, spearman, pearson, kendall_tau

human, metric = make_correlation_data()
rho = spearman(metric, human)
r = pearson(metric, human)
tau = kendall_tau(metric, human)

# Cross-check our hand-rolled Spearman against scipy (must agree).
from scipy.stats import spearmanr
rho_scipy = spearmanr(metric, human).statistic
assert abs(rho - rho_scipy) < 1e-9, f'our Spearman {rho} must match scipy {rho_scipy}'
assert 0.7 < rho < 1.0, f'Spearman should be high-but-imperfect, got {rho}'

print(f'Spearman rho = {rho:.3f}   (scipy check: {rho_scipy:.3f} — match)')
print(f'Pearson  r   = {r:.3f}')
print(f'Kendall  tau = {tau:.3f}')
print('-> high but < 1: good for RANKING systems, unreliable per single sentence')

Spearman rho = 0.939   (scipy check: 0.939 — match)
Pearson  r   = 0.931
Kendall  tau = 0.822
-> high but < 1: good for RANKING systems, unreliable per single sentence


![Correlation](../../images/nlpeval_correlation.png)

*Metric vs human: points cluster around the diagonal (perfect agreement) with scatter — ρ ≈ 0.94. Use such a metric for fast iteration and system comparison, never to accept or reject one output.*

## 7. Library cross-check — our from-scratch math vs the standard tools

Finally, prove the from-scratch implementations are **correct**, not just internally consistent: our BLEU must match **sacreBLEU** and our ROUGE-L must match Google's **rouge-score**, to the digit (feeding the libraries the same whitespace-tokenized text so we compare the *math*, not the tokenization).

In [9]:
from nlp_evaluation import library_crosscheck, sentence_bleu, rouge_l

cross = library_crosscheck()
if cross is None:
    print('sacrebleu / rouge_score not installed — skipping (the core math above still ran)')
else:
    ours_bleu = sentence_bleu('the cat sat on the mat', 'the cat sat on the warm mat')['bleu']
    ours_rl = rouge_l('the cat was sitting on the mat', 'the cat sat on the mat')['f']
    assert abs(cross['lib_bleu'] - ours_bleu) < 0.01, 'our BLEU must match sacreBLEU'
    assert abs(cross['lib_rougeL_f'] - ours_rl) < 1e-6, 'our ROUGE-L must match rouge-score'
    print(f"our BLEU    {ours_bleu:7.3f}  ==  sacreBLEU   {cross['lib_bleu']:7.3f}   (match)")
    print(f"our ROUGE-L {ours_rl:7.3f}  ==  rouge-score {cross['lib_rougeL_f']:7.3f}   (match)")

our BLEU     67.318  ==  sacreBLEU    67.318   (match)
our ROUGE-L   0.769  ==  rouge-score   0.769   (match)


## Recap

Everything asserted, then printed, every number reproducible:

- **EM vs token-F1** — strict 0 vs forgiving 0.80; QA reports both.
- **BLEU = 67.3** from scratch, matching sacreBLEU to the digit.
- **The metric zoo** — five 'overlap' metrics give wildly different scores on one pair.
- **Meaning-blindness** — a paraphrase scores BLEU ≈ 0 but semantic ≈ 86; and the embedding metric is **fooled by negation**.
- **Bootstrap** — a +2.86 'win' whose 95% CI straddles zero is **not** significant.
- **Correlation** — Spearman ρ ≈ 0.94 (matched by scipy): trust a metric for ranking, never for a single verdict.

The through-line: **every automatic metric is a proxy for a human, so metric–human correlation is the meta-metric, and a single number without a significance test is a claim you have not earned.**